# Install Libraries

In [25]:
!pip install groq python-dotenv scikit-learn pandas --quiet

In [ ]:
from groq import Groq
from dotenv import load_dotenv

import os


In [39]:
load_dotenv(override=True)

api_key = os.getenv("GROQ_API_KEY")

print(api_key[:4] + "...")

gsk_...


In [4]:
client = Groq(
    api_key=api_key
)

print("Client initialized")

Client initialized


In [5]:
INTENTS = [
    "greeting",
    "goodbye",
    "gratitude",
    "asking_mental_health_question",
    "out_of_scope"
]

print(INTENTS)

['greeting', 'goodbye', 'gratitude', 'asking_mental_health_question', 'out_of_scope']


In [6]:
prompt = """
Classify the user's intent.

Possible intents:
1. greeting
2. goodbye
3. gratitude
4. asking_mental_health_question
5. out_of_scope

Return ONLY the intent label.

User:
Hello there
"""

print(prompt)


Classify the user's intent.

Possible intents:
1. greeting
2. goodbye
3. gratitude
4. asking_mental_health_question
5. out_of_scope

Return ONLY the intent label.

User:
Hello there



# Zero Shot

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

greeting


In [12]:
test_questions = [
    "Hello",
    "Good morning",
    "Thank you very much",
    "Bye",
    "Can anxiety affect sleep?",
    "Who won the world cup?"
]

for question in test_questions:

    prompt = f"""
    Classify the user's intent.

    Possible intents:
    greeting
    goodbye
    gratitude
    asking_mental_health_question
    out_of_scope

    Return ONLY the label.

    User:
    {question}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()

    print(f"Question: {question}")
    print(f"Prediction: {prediction}")
    print("-"*50)

Question: Hello
Prediction: greeting
--------------------------------------------------
Question: Good morning
Prediction: greeting
--------------------------------------------------
Question: Thank you very much
Prediction: gratitude
--------------------------------------------------
Question: Bye
Prediction: goodbye
--------------------------------------------------
Question: Can anxiety affect sleep?
Prediction: asking_mental_health_question
--------------------------------------------------
Question: Who won the world cup?
Prediction: out_of_scope
--------------------------------------------------


In [14]:
edge_cases = [
    "hey",
    "see you later",
    "thanks",
    "I appreciate your help",
    "I'm feeling anxious all the time",
    "What is the capital of France?",
    "Can stress cause headaches?",
    "Good night",
    "Thank you so much",
    "How do I install Python?"
]

for question in edge_cases:

    prompt = f"""
    Classify the user's intent.

    Possible intents:
    greeting
    goodbye
    gratitude
    asking_mental_health_question
    out_of_scope

    Return ONLY the label.

    User:
    {question}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()

    print(f"Question: {question}")
    print(f"Prediction: {prediction}")
    print("-"*50)

Question: hey
Prediction: greeting
--------------------------------------------------
Question: see you later
Prediction: goodbye
--------------------------------------------------
Question: thanks
Prediction: gratitude
--------------------------------------------------
Question: I appreciate your help
Prediction: gratitude
--------------------------------------------------
Question: I'm feeling anxious all the time
Prediction: asking_mental_health_question
--------------------------------------------------
Question: What is the capital of France?
Prediction: out_of_scope
--------------------------------------------------
Question: Can stress cause headaches?
Prediction: asking_mental_health_question
--------------------------------------------------
Question: Good night
Prediction: goodbye
--------------------------------------------------
Question: Thank you so much
Prediction: gratitude
--------------------------------------------------
Question: How do I install Python?
Prediction:

In [15]:
SYSTEM_PROMPT = """
You are an intent classification engine.

Your task is to classify a user message into exactly ONE of these labels:

greeting
goodbye
gratitude
asking_mental_health_question
out_of_scope

Definitions:

greeting:
The user is saying hello.

goodbye:
The user is ending the conversation.

gratitude:
The user is expressing thanks or appreciation.

asking_mental_health_question:
The user is asking about mental health, anxiety, depression, stress, emotions, coping strategies, wellbeing, counseling, therapy, or related topics.

out_of_scope:
Anything unrelated to mental health.

Return ONLY the label.
"""

In [16]:
def classify_intent_zero_shot(question):

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.choices[0].message.content.strip()

In [19]:
for q in edge_cases:
    pred = classify_intent_zero_shot(q)

    print(f"Question: {q}")
    print(f"Prediction: {pred}")
    print("-"*50)

Question: hey
Prediction: greeting
--------------------------------------------------
Question: see you later
Prediction: goodbye
--------------------------------------------------
Question: thanks
Prediction: gratitude
--------------------------------------------------
Question: I appreciate your help
Prediction: gratitude
--------------------------------------------------
Question: I'm feeling anxious all the time
Prediction: asking_mental_health_question
--------------------------------------------------
Question: What is the capital of France?
Prediction: out_of_scope
--------------------------------------------------
Question: Can stress cause headaches?
Prediction: asking_mental_health_question
--------------------------------------------------
Question: Good night
Prediction: goodbye
--------------------------------------------------
Question: Thank you so much
Prediction: gratitude
--------------------------------------------------
Question: How do I install Python?
Prediction:

# Few Shot

In [20]:
FEW_SHOT_EXAMPLES = """
User: Hello
Intent: greeting

User: Hi there
Intent: greeting

User: Goodbye
Intent: goodbye

User: See you later
Intent: goodbye

User: Thanks
Intent: gratitude

User: I appreciate your help
Intent: gratitude

User: Can anxiety affect sleep?
Intent: asking_mental_health_question

User: How can I manage stress?
Intent: asking_mental_health_question

User: Who won the football match?
Intent: out_of_scope

User: How do I install Python?
Intent: out_of_scope
"""

FEW_SHOT_SYSTEM_PROMPT = f"""
You are an intent classifier.

Possible intents:

greeting
goodbye
gratitude
asking_mental_health_question
out_of_scope

Examples:

{FEW_SHOT_EXAMPLES}

Return ONLY the intent label.
"""

In [21]:
def classify_intent_few_shot(question):

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": FEW_SHOT_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.choices[0].message.content.strip()

# Comparison Between Zero Shot and Few Shot

In [ ]:
for q in edge_cases:

    zero_pred = classify_intent_zero_shot(q)

    few_pred = classify_intent_few_shot(q)

    print(f"Question: {q}")
    print(f"Zero-shot : {zero_pred}")
    print(f"Few-shot  : {few_pred}")
    print("-"*50)

Question: hey
Zero-shot : greeting
Few-shot  : greeting
------------------------------------------------------------
Question: see you later
Zero-shot : goodbye
Few-shot  : goodbye
------------------------------------------------------------
Question: thanks
Zero-shot : gratitude
Few-shot  : gratitude
------------------------------------------------------------
Question: I appreciate your help
Zero-shot : gratitude
Few-shot  : gratitude
------------------------------------------------------------
Question: I'm feeling anxious all the time
Zero-shot : asking_mental_health_question
Few-shot  : asking_mental_health_question
------------------------------------------------------------
Question: What is the capital of France?
Zero-shot : out_of_scope
Few-shot  : out_of_scope
------------------------------------------------------------
Question: Can stress cause headaches?
Zero-shot : asking_mental_health_question
Few-shot  : asking_mental_health_question
------------------------------------

# Structured Output Function

In [36]:
def predict_intent(question):

    intent = classify_intent_few_shot(question)

    return {
        "text": question,
        "intent": intent
    }

In [37]:
predict_intent(
    "I feel anxious lately"
)

{'text': 'I feel anxious lately', 'intent': 'asking_mental_health_question'}

# Simulate Chatbot Routing

In [38]:
query = "Can anxiety affect sleep?"

result = predict_intent(query)

intent = result["intent"]

if intent == "greeting":
    print("Greeting Route")

elif intent == "goodbye":
    print("Goodbye Route")

elif intent == "gratitude":
    print("Gratitude Route")

elif intent == "out_of_scope":
    print("Out Of Scope Route")

elif intent == "asking_mental_health_question":
    print("RAG Route")

RAG Route
